In [ ]:
import requests
import time
from notebookutils import mssparkutils
from datetime import datetime, timezone, timedelta
from pyspark.sql.functions import max as spark_max
from pyspark.sql.types import StructType, StructField, StringType, BooleanType, TimestampType

# --- Configuration ---
kusto_cluster_uri = "https://trd-6uegjpfbf030eemxtw.z1.kusto.fabric.microsoft.com"
kusto_database = "MonitoringEventhouse"

tenant_id = "9e929790-272d-4977-a2ab-301443c11ece"
client_id = "b5c04c9c-0588-418f-8f60-2d83d38cb635"
client_secret = "your-secret"
# client_secret = mssparkutils.credentials.getSecret("your-kv", "your-secret")

fabric_api_base = "https://api.fabric.microsoft.com/v1"
pbi_api_base    = "https://api.powerbi.com/v1.0/myorg"
fix_aop_config_table = "Fix_AOP_Config"

StatementMeta(, d85afa0f-a4a3-4157-b698-7e52b7039c16, 3, Finished, Available, Finished, False)

In [3]:
# --- FOR TESTING/DEVELOPMENT ONLY ---
# This cell manually resets the watermark to 10 days ago to reprocess historical alerts.
# Comment out or remove this cell for normal scheduled/production runs.
# from datetime import timedelta

# ten_days_ago = (datetime.now(timezone.utc) - timedelta(days=10)).strftime("%Y-%m-%dT%H:%M:%S")
# spark.sql(f"UPDATE {fix_aop_config_table} SET LastRunTimestamp = CAST('{ten_days_ago}' AS TIMESTAMP), UpdatedAt = current_timestamp()")
# print(f"LastRunTimestamp set to: {ten_days_ago}")


StatementMeta(, d85afa0f-a4a3-4157-b698-7e52b7039c16, 5, Finished, Available, Finished, False)

In [ ]:
def get_spn_token(scope):
    url = f"https://login.microsoftonline.com/{tenant_id}/oauth2/v2.0/token"
    payload = {
        'grant_type': 'client_credentials',
        'client_id': client_id,
        'client_secret': client_secret,
        'scope': scope
    }
    try:
        response = requests.post(url, data=payload)
        response.raise_for_status()
        return response.json().get("access_token")
    except Exception as e:
        print(f"Failed to retrieve token for scope {scope}: {e}")
        return None


def get_tokens():
    global fabric_token, kusto_token
    fabric_token = get_spn_token("https://api.fabric.microsoft.com/.default")
    kusto_token  = get_spn_token(f"{kusto_cluster_uri}/.default")
    if fabric_token and kusto_token:
        print("Tokens retrieved.")
    else:
        raise Exception("ERROR: Failed to retrieve one or more tokens.")


StatementMeta(, d85afa0f-a4a3-4157-b698-7e52b7039c16, 6, Finished, Available, Finished, False)

Tokens retrieved.


In [ ]:
def load_watermark():
    watermark_ts = None
    df_config = spark.table(fix_aop_config_table)
    max_row = df_config.agg(spark_max("LastRunTimestamp").alias("max_ts")).collect()
    max_ts = max_row[0]["max_ts"]

    if max_ts:
        watermark_ts = max_ts
        if watermark_ts.tzinfo is None:
            watermark_ts = watermark_ts.replace(tzinfo=timezone.utc)
        print(f"Watermark loaded: {watermark_ts}")
    else:
        print("No watermark in Fix_AOP_Config — will process all records.")

    return watermark_ts


StatementMeta(, d85afa0f-a4a3-4157-b698-7e52b7039c16, 7, Finished, Available, Finished, False)

Watermark loaded: 2026-04-09 06:46:01+00:00


In [ ]:
def get_workspaces_to_fix(upper_bound, lower_bound=None):
    upper_bound_str = upper_bound.strftime("%Y-%m-%dT%H:%M:%SZ")

    if lower_bound is not None:
        # Loop iteration: use explicit lower bound from previous run
        lower_bound_str = lower_bound.strftime("%Y-%m-%dT%H:%M:%SZ")
        watermark_filter = f'| where ingestion_time() > datetime("{lower_bound_str}") and ingestion_time() <= datetime("{upper_bound_str}")'
    else:
        # Initial run: derive lower bound from watermark table
        watermark_ts = load_watermark()
        if watermark_ts:
            watermark_str = watermark_ts.strftime("%Y-%m-%dT%H:%M:%SZ")
            watermark_filter = f'| where ingestion_time() > datetime("{watermark_str}") and ingestion_time() <= datetime("{upper_bound_str}")'
        else:
            watermark_filter = f'| where ingestion_time() <= datetime("{upper_bound_str}")'

    kusto_query = f"""
    AOPAlertLogs
    {watermark_filter}
    | summarize arg_max(ingestion_time(), *) by WorkspaceId
    | where AlertStatus in ("EmailSent", "NoEmail")
    | where ToChangeAOP == true
    | project WorkspaceId, AOP, AOPMustBe
    """
    
    try:
        df = spark.read \
            .format("com.microsoft.kusto.spark.synapse.datasource") \
            .option("kustoCluster", kusto_cluster_uri) \
            .option("kustoDatabase", kusto_database) \
            .option("kustoQuery", kusto_query) \
            .option("accessToken", kusto_token) \
            .load()

        workspaces = df.collect()
        print(f"Found {len(workspaces)} workspace(s) to fix.")
        return workspaces
    except Exception as e:
        print(f"Error reading from Kusto: {e}")
        return []


StatementMeta(, d85afa0f-a4a3-4157-b698-7e52b7039c16, 8, Finished, Available, Finished, False)

Found 0 workspace(s) to fix.


In [7]:
def get_workspace_comm_policy(workspace_id):
    url = f"{fabric_api_base}/workspaces/{workspace_id}/networking/communicationPolicy"
    headers = {"Authorization": f"Bearer {fabric_token}"}
    response = requests.get(url, headers=headers)
    response.raise_for_status()
    return response.json(), response.headers.get("ETag")


def set_workspace_aop(workspace_id, aop_must_be):
    """
    Applies AOP policy to a workspace.
    Returns (status, detail, error_code):
      - Success: ("Success", outbound_action, None)
      - Failure: ("Failed", error_description, error_code_str)
    """
    outbound_action = "Deny" if aop_must_be == "EnableWorkspaceOutboundAccessProtection" else "Allow"

    try:
        current_policy, etag = get_workspace_comm_policy(workspace_id)
    except Exception as e:
        return "Failed", f"GET policy failed: {e}", "GetPolicyFailed"

    if "outbound" not in current_policy:
        current_policy["outbound"] = {}
    current_policy["outbound"]["publicAccessRules"] = {"defaultAction": outbound_action}

    url = f"{fabric_api_base}/workspaces/{workspace_id}/networking/communicationPolicy"
    headers = {"Authorization": f"Bearer {fabric_token}", "Content-Type": "application/json"}
    if etag:
        headers["If-Match"] = etag

    response = requests.put(url, headers=headers, json=current_policy)

    if response.status_code == 429:
        retry_after = int(response.headers.get("Retry-After", 60))
        print(f"  Rate limited. Waiting {retry_after}s...")
        time.sleep(retry_after)
        response = requests.put(url, headers=headers, json=current_policy)

    if response.status_code == 200:
        return "Success", outbound_action, None

    error_code = ""
    try:
        error_code = response.json().get("errorCode", "")
    except Exception:
        pass

    return "Failed", f"{response.status_code}: {response.text}", error_code


StatementMeta(, d85afa0f-a4a3-4157-b698-7e52b7039c16, 9, Finished, Available, Finished, False)

In [8]:
# Item types that are compatible with AOP — never deleted
ALLOWED_ITEM_TYPES = {
    "Lakehouse", 
    "Notebook", 
    "SparkJobDefinition", 
    "Environment",
    "Warehouse", 
    "DataFlow", 
    "DataPipeline", 
    "CopyJob",
    "MirroredDatabase", 
    "SQLEndpoint", 
    "VariableLibrary", 
    "SqlAnalyticsEndpoint",
    "Experiment", # no events
    "MlModel", # no events
}

# PBI item types that block AOP enablement, in required deletion order
PBI_DELETION_ORDER = [
    "Scorecard", "Exploration", "Dashboard", "App",
    "PaginatedReport", "Report", "SemanticModel",
]

# Item types that must be deleted via the Power BI REST API instead of Fabric API
PBI_ENDPOINT_TYPES = {"Report", "PaginatedReport", "Dashboard"}

# Maps item type → REST API endpoint segment
ITEM_ENDPOINT_MAP = {
    "ApacheAirflowJob":              "ApacheAirflowJobs",
    "CopyJob":                       "copyJobs",
    "CosmosDBDatabase":              "cosmosDbDatabases",
    "Dashboard":                     "dashboards",
    "DataAgent":                     "dataAgents",
    "DataFlow":                      "dataflows",
    "DataPipeline":                  "dataPipelines",
    "Environment":                   "environments",
    "Eventhouse":                    "eventhouses",
    "EventSchemaSet":                "eventSchemaSets",
    "GraphQLApi":                    "GraphQLApis",
    "KQLDashboard":                  "kqlDashboards",
    "KQLQueryset":                   "kqlQuerysets",
    "Lakehouse":                     "lakehouses",
    "Map":                           "maps",
    "MirroredAzureDatabricksCatalog": "mirroredAzureDatabricksCatalogs",
    "MirroredDatabase":              "mirroredDatabases",
    "MountedDataFactory":            "mountedDataFactories",
    "Notebook":                      "notebooks",
    "PaginatedReport":               "reports",
    "Reflex":                        "reflexes",
    "Report":                        "reports",
    "SemanticModel":                 "semanticModels",
    "SparkJobDefinition":            "sparkJobDefinitions",
    "SQLDatabase":                   "sqlDatabases",
    "UserDataFunction":              "UserDataFunctions",
    "VariableLibrary":               "VariableLibraries",
    "Warehouse":                     "warehouses",
}

alert_logs_schema = StructType([
    StructField("WorkspaceName", StringType(),    True),
    StructField("WorkspaceId",   StringType(),    True),
    StructField("wstime",        TimestampType(), True),
    StructField("data_itemKind", StringType(),    True),
    StructField("data_itemName", StringType(),    True),
    StructField("data_itemId",   StringType(),    True),
    StructField("AlertStatus",   StringType(),    True),
])


def write_alert_logs(log_entries):
    if not log_entries:
        return
    df = spark.createDataFrame(log_entries, schema=alert_logs_schema)
    df.write \
        .format("com.microsoft.kusto.spark.synapse.datasource") \
        .option("kustoCluster", kusto_cluster_uri) \
        .option("kustoDatabase", kusto_database) \
        .option("kustoTable", "AlertLogs") \
        .option("accessToken", kusto_token) \
        .option("tableCreateOptions", "CreateIfNotExist") \
        .mode("Append") \
        .save()
    print(f"  Written {len(log_entries)} AlertLogs entry(ies).")


def get_workspace_name(workspace_id):
    url = f"{fabric_api_base}/workspaces/{workspace_id}"
    headers = {"Authorization": f"Bearer {fabric_token}"}
    response = requests.get(url, headers=headers)
    if response.status_code == 200:
        return response.json().get("displayName", workspace_id)
    return workspace_id


def get_workspace_items(workspace_id):
    """
    Lists all items in a workspace, handling pagination.
    API: GET /v1/workspaces/{workspaceId}/items
    Note: does not include Scorecards — use get_scorecards_pbi for those.
    """
    url = f"{fabric_api_base}/workspaces/{workspace_id}/items"
    headers = {"Authorization": f"Bearer {fabric_token}"}
    items = []
    while url:
        response = requests.get(url, headers=headers)
        response.raise_for_status()
        data = response.json()
        items.extend(data.get("value", []))
        url = data.get("continuationUri")
    return items


def get_scorecards_pbi(workspace_id):
    """
    Lists all scorecards in a workspace via the Power BI REST API.
    API: GET {pbi_api_base}/groups/{workspaceId}/scorecards
    Returns a list of (id, name) tuples.
    """
    url = f"{pbi_api_base}/groups/{workspace_id}/scorecards"
    headers = {"Authorization": f"Bearer {fabric_token}"}
    scorecards = []
    while url:
        response = requests.get(url, headers=headers)
        response.raise_for_status()
        data = response.json()
        scorecards.extend((item["id"], item.get("name", item["id"])) for item in data.get("value", []))
        url = data.get("@odata.nextLink")
    return scorecards


def delete_scorecard_pbi(workspace_id, item_id):
    """
    Deletes a single scorecard via the Power BI REST API (OData URL style).
    API: DELETE {pbi_api_base}/groups/{workspaceId}/scorecards({scorecardId})
    Returns True on success or 404, False otherwise.
    """
    url = f"{pbi_api_base}/groups/{workspace_id}/scorecards({item_id})"
    headers = {"Authorization": f"Bearer {fabric_token}", "Content-Type": "application/json"}
    response = requests.delete(url, headers=headers)
    if response.status_code in (200, 204, 404):
        return True
    print(f"    Scorecard delete failed for {item_id}: {response.status_code} {response.text}")
    return False


def get_blocking_items(workspace_id):
    """Returns items that prevent AOP enablement (not in ALLOWED_ITEM_TYPES)."""
    all_items = get_workspace_items(workspace_id)
    return [item for item in all_items if item.get("type") not in ALLOWED_ITEM_TYPES]


def delete_item(workspace_id, item_id, item_type, max_retries=2):
    """
    Deletes a single item.
    - Report, PaginatedReport, Dashboard → Power BI REST API
      DELETE {pbi_api_base}/groups/{workspaceId}/{endpoint}/{itemId}
    - All other types → Fabric REST API
      DELETE {fabric_api_base}/workspaces/{workspaceId}/{endpoint}/{itemId}
    Returns True on success or 404 (already gone), False otherwise.
    """
    endpoint = ITEM_ENDPOINT_MAP.get(item_type, "items")

    if item_type in PBI_ENDPOINT_TYPES:
        url = f"{pbi_api_base}/groups/{workspace_id}/{endpoint}/{item_id}"
    else:
        url = f"{fabric_api_base}/workspaces/{workspace_id}/{endpoint}/{item_id}"

    headers = {"Authorization": f"Bearer {fabric_token}", "Content-Type": "application/json"}

    for _ in range(max_retries):
        response = requests.delete(url, headers=headers)
        if response.status_code in (200, 204):
            return True
        if response.status_code == 404:
            return True  # Already gone
        if response.status_code == 429:
            retry_after = int(response.headers.get("Retry-After", 60))
            print(f"    Rate limited. Waiting {retry_after}s...")
            time.sleep(retry_after)
            continue
        print(f"    Delete failed for {item_id}: {response.status_code} {response.text}")
        return False

    print(f"    Max retries exceeded for {item_id}.")
    return False


def delete_blocking_items(workspace_id, workspace_name, blocking_items):
    """
    Deletes blocking items in the correct order:
      1. PBI types in dependency order (Scorecard → … → SemanticModel)
      2. All remaining non-allowed types
    Returns (failed_items, log_entries).
    """
    pbi_buckets = {kind: [] for kind in PBI_DELETION_ORDER}
    other_items = []

    for item in blocking_items:
        item_type = item.get("type", "")
        if item_type in pbi_buckets:
            pbi_buckets[item_type].append(item)
        else:
            other_items.append(item)

    failed = []
    log_entries = []

    for kind in PBI_DELETION_ORDER:
        for item in pbi_buckets[kind]:
            name = item.get("displayName", item["id"])
            print(f"    Deleting {kind} '{name}' ({item['id']})")
            success = delete_item(workspace_id, item["id"], kind)
            log_entries.append((
                workspace_name, workspace_id, datetime.now(timezone.utc),
                kind, name, item["id"],
                "DeletedItem" if success else "DeleteFailed",
            ))
            if not success:
                failed.append(item)

    for item in other_items:
        item_type = item.get("type", "unknown")
        name = item.get("displayName", item["id"])
        print(f"    Deleting {item_type} '{name}' ({item['id']})")
        success = delete_item(workspace_id, item["id"], item_type)
        log_entries.append((
            workspace_name, workspace_id, datetime.now(timezone.utc),
            item_type, name, item["id"],
            "DeletedItem" if success else "DeleteFailed",
        ))
        if not success:
            failed.append(item)

    return failed, log_entries


def clear_blocking_items_and_retry_aop(workspace_id, aop_must_be):
    """
    Resolves OutboundRestrictionNotEligible by:
      1. Deleting scorecards via PBI API (not visible in Fabric items list)
      2. Deleting remaining blocking items via Fabric/PBI API
      3. Writing all deletion results to AlertLogs
      4. Retrying set_workspace_aop
    Returns same (status, detail, error_code) signature as set_workspace_aop.
    """
    print(f"  Resolving OutboundRestrictionNotEligible for {workspace_id}...")

    workspace_name = get_workspace_name(workspace_id)
    all_log_entries = []

    # Step 1: delete scorecards (not returned by get_workspace_items)
    try:
        scorecard_ids = get_scorecards_pbi(workspace_id)
    except Exception as e:
        return "Failed", f"Could not list scorecards: {e}", "GetScorecardsFailed"

    if scorecard_ids:
        print(f"  Found {len(scorecard_ids)} scorecard(s). Deleting...")
        failed_scorecards = []
        for sid, sname in scorecard_ids:
            success = delete_scorecard_pbi(workspace_id, sid)
            all_log_entries.append((
                workspace_name, workspace_id, datetime.now(timezone.utc),
                "Scorecard", sname, sid,
                "DeletedItem" if success else "DeleteFailed",
            ))
            if not success:
                failed_scorecards.append(sid)
        if failed_scorecards:
            write_alert_logs(all_log_entries)
            return "Failed", f"Could not delete scorecard(s): {failed_scorecards}", "ScorecardDeletionFailed"

    # Step 2: delete remaining blocking items from Fabric items list
    try:
        blocking_items = get_blocking_items(workspace_id)
    except Exception as e:
        write_alert_logs(all_log_entries)
        return "Failed", f"Could not list workspace items: {e}", "GetItemsFailed"

    if not blocking_items and not scorecard_ids:
        return "Failed", "No blocking items found but AOP still failed", "OutboundRestrictionNotEligible"

    if blocking_items:
        print(f"  Found {len(blocking_items)} blocking item(s). Deleting...")
        failed_items, log_entries = delete_blocking_items(workspace_id, workspace_name, blocking_items)
        all_log_entries.extend(log_entries)
        if failed_items:
            write_alert_logs(all_log_entries)
            names = ", ".join(f"{i.get('type')} '{i.get('displayName')}'" for i in failed_items)
            return "Failed", f"Could not delete: {names}", "ItemDeletionFailed"

    write_alert_logs(all_log_entries)
    print("  All blocking items deleted. Retrying AOP change...")
    return set_workspace_aop(workspace_id, aop_must_be)


StatementMeta(, d85afa0f-a4a3-4157-b698-7e52b7039c16, 10, Finished, Available, Finished, False)

In [ ]:
results_schema = StructType([
    StructField("WorkspaceId",  StringType(),    True),
    StructField("ToChangeAOP",  BooleanType(),   True),
    StructField("AOP",          StringType(),    True),
    StructField("AOPMustBe",    StringType(),    True),
    StructField("CreationTime", TimestampType(), True),
    StructField("UserId",       StringType(),    True),
    StructField("AlertStatus",  StringType(),    True),
])


def process_workspaces(workspaces):
    fix_results = []

    if not workspaces:
        print("No workspaces to process.")
        return fix_results

    if not fabric_token:
        raise Exception("Aborting: No Fabric token available.")

    for row in workspaces:
        ws_id       = row["WorkspaceId"]
        aop_current = row["AOP"]
        aop_value   = row["AOPMustBe"]
        print(f"Setting AOP for {ws_id} → {aop_value}")
        api_call_time = datetime.now(timezone.utc)

        status, detail, error_code = set_workspace_aop(ws_id, aop_value)

        # If AOP cannot be enabled because incompatible items exist, remove them and retry
        if (status == "Failed"
                and error_code == "OutboundRestrictionNotEligible"
                and aop_value == "EnableWorkspaceOutboundAccessProtection"):
            status, detail, error_code = clear_blocking_items_and_retry_aop(ws_id, aop_value)

        print(f"  {status}: {detail}")

        if status == "Success":
            fix_results.append((
                ws_id,
                False,          # ToChangeAOP — fixed, no longer needs changing
                aop_value,      # AOP — new value equals AOPMustBe
                aop_value,      # AOPMustBe
                api_call_time,
                "system",
                "Fixed",
            ))
        else:
            alert_status = f"Error:{error_code}" if error_code else f"Error:{detail}"
            fix_results.append((
                ws_id,
                True,           # ToChangeAOP — still needs fixing
                aop_current,    # AOP — unchanged, original value from AOPAlertLogs
                aop_value,      # AOPMustBe
                api_call_time,
                "system",
                alert_status,
            ))

    if fix_results:
        results_df = spark.createDataFrame(fix_results, schema=results_schema)
        try:
            results_df.write \
                .format("com.microsoft.kusto.spark.synapse.datasource") \
                .option("kustoCluster", kusto_cluster_uri) \
                .option("kustoDatabase", kusto_database) \
                .option("kustoTable", "AOPAlertLogs") \
                .option("accessToken", kusto_token) \
                .option("tableCreateOptions", "CreateIfNotExist") \
                .mode("Append") \
                .save()
            print(f"Written {len(fix_results)} fix result(s) to AOPAlertLogs.")
        except Exception as e:
            print(f"Failed to write fix results to Eventhouse: {e}")
    else:
        print("No fix results to write back.")

    return fix_results


StatementMeta(, d85afa0f-a4a3-4157-b698-7e52b7039c16, 11, Finished, Available, Finished, False)

No workspaces to process.
No fix results to write back.


In [ ]:
def update_watermark(timestamp):
    new_watermark_str = timestamp.strftime("%Y-%m-%dT%H:%M:%S")
    try:
        spark.sql(f"""
            MERGE INTO {fix_aop_config_table} AS t
            USING (SELECT CAST('{new_watermark_str}' AS TIMESTAMP) AS LastRunTimestamp) AS s
            ON (1 = 1)
            WHEN MATCHED THEN
                UPDATE SET t.LastRunTimestamp = s.LastRunTimestamp, t.UpdatedAt = current_timestamp()
            WHEN NOT MATCHED THEN
                INSERT (LastRunTimestamp, UpdatedAt) VALUES (s.LastRunTimestamp, current_timestamp())
        """)
        print(f"Watermark updated to: {new_watermark_str}")
    except Exception as e:
        print(f"Failed to update watermark: {e}")


StatementMeta(, d85afa0f-a4a3-4157-b698-7e52b7039c16, 12, Finished, Available, Finished, False)

Watermark updated to: 2026-04-09T07:12:49


In [ ]:
# --- Main Execution ---
get_tokens()

# Initial run: process all records since last watermark
upper_bound = datetime.now(timezone.utc)
workspaces  = get_workspaces_to_fix(upper_bound)
if workspaces:
    process_workspaces(workspaces)
update_watermark(upper_bound)

# --- Continuous Polling: run every minute for 1 hour ---
# Each iteration re-queries for new records since the last run
# Watermark is updated only once after the final iteration
polling_duration_hours   = 1
polling_interval_seconds = 60

loop_start_time = datetime.now(timezone.utc)
loop_end_time   = loop_start_time + timedelta(hours=polling_duration_hours)

print(f"\nStarting continuous polling until {loop_end_time.isoformat()}")

iteration = 0

while True:
    now = datetime.now(timezone.utc)
    if now >= loop_end_time:
        print(f"\nPolling duration of {polling_duration_hours}h reached. Stopping.")
        break

    iteration += 1

    # Slide the window: previous upper_bound becomes lower_bound
    lower_bound = upper_bound
    upper_bound = datetime.now(timezone.utc)

    print(f"\n[Iteration {iteration}] {upper_bound.strftime('%H:%M:%S UTC')} | window: [{lower_bound.strftime('%H:%M:%S')} - {upper_bound.strftime('%H:%M:%S')}]")

    workspaces = get_workspaces_to_fix(upper_bound, lower_bound=lower_bound)
    if workspaces:
        process_workspaces(workspaces)

    if datetime.now(timezone.utc) < loop_end_time:
        print(f"Sleeping {polling_interval_seconds}s...")
        time.sleep(polling_interval_seconds)

# Update watermark once after the final iteration
update_watermark(upper_bound)

print("Continuous polling finished.")